# Set Ups

In [3]:
import os
# import uuid
import nest_asyncio
import asyncio
from fastapi import UploadFile
from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.extractors import TitleExtractor
from llama_index.core.text_splitter import SentenceSplitter
from llama_index.readers.docling import DoclingReader
from llama_index.core.schema import Document
from dotenv import load_dotenv
load_dotenv()

from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb
from llama_index.llms.ollama import Ollama
import torch
from llama_index.core.extractors import SummaryExtractor, KeywordExtractor
from pathlib import Path
nest_asyncio.apply()

/Users/beckyxu/Documents/GitHub/Carbon_Offset_Validation/carbon-offset-validator/server-python/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
if torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
    
def llm_setting(name = "Local"):
    if name == "Gemini": 
        from llama_index.llms.gemini import Gemini
        from llama_index.embeddings.gemini import GeminiEmbedding
        from llama_index.embeddings.huggingface import HuggingFaceEmbedding
        api_key = os.getenv("GEMINI_API_KEY")
        # define embedding model
        embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5", 
                                           device=device,
                                           embed_batch_size=10)
        llm = Gemini(model_name="models/gemini-1.5-flash", temperature=0.1, max_tokens=50000, api_key=api_key)
    
    # if default -> use local model with HF embedding
    else:
        # api_key = os.getenv("HF_API_KEY")
        # define embedding model
        from llama_index.llms.ollama import Ollama
        from llama_index.embeddings.huggingface import HuggingFaceEmbedding
        embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5", 
                                           device=device,
                                           embed_batch_size=10)
        
        llm = Ollama(model="deepseek-r1:7b", 
                     request_timeout=120.0)
    return llm, embed_model

In [6]:
# Use DoclingReader to load the data
reader = DoclingReader()

# Set up model
llm, embed_model = llm_setting()

# Set default LLM and embedding model 
Settings.llm = llm
Settings.embed_model = embed_model


# 1. PDF to LLamaIndex in ChromaDB - PDD & Policy

## Global setting

In [192]:
from llama_index.core.ingestion import IngestionPipeline, IngestionCache
# file_paths = "/Users/beckyxu/Documents/GitHub/Carbon_Offset_Validation/carbon-offset-validator/server-python/test/Carbon_Market.pdf"

### Global Setting
chroma_client = chromadb.PersistentClient(path="./chroma_db_new")
reader = DoclingReader()
node_parser = MarkdownNodeParser()


## PDD Process

In [205]:
### PDD
# NOTE: Change things here
file_path_pdd = 'pdd/VCS_3226_302541_Padang Tikar_REDD_PDD_VCS_v1.0_clean (1).pdf'

file_name = os.path.basename(file_path)
chroma_collection_pdd = chroma_client.get_or_create_collection(name="pdd")# construct vector store and customize storage context
storage_context_pdd  = StorageContext.from_defaults(
    vector_store=ChromaVectorStore(chroma_collection_pdd) 
)
documents = reader.load_data(Path(file_path_pdd))
for doc in documents:
    if doc.metadata is None:
        doc.metadata = {}
    doc.metadata.update({
        "file_name": file_name,
        "registry": file_name.split("_")[0],
        "project_code": file_name.split("_")[1],
        "type": "pdd"
    })
    
index_pdd = VectorStoreIndex.from_documents(
    documents=documents,
    transformations=[node_parser],
    storage_context=storage_context_pdd,
    cache=IngestionCache()
)


INFO:docling.document_converter:Going to convert document batch...
INFO:docling.pipeline.base_pipeline:Processing document VCS_3226_302541_Padang Tikar_REDD_PDD_VCS_v1.0_clean (1).pdf
INFO:docling.document_converter:Finished converting document VCS_3226_302541_Padang Tikar_REDD_PDD_VCS_v1.0_clean (1).pdf in 158.62 sec.
Batches: 100%|██████████| 1/1 [00:00<00:00,  4.69it/s]


## Policy VCM Process

In [204]:

### POLICY VCM document

# get folder of the policy docs
def get_file_paths(folder_path):
    """Get all file paths in a folder recursively, excluding .DS_Store files."""
    file_paths = []
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file != '.DS_Store':
                file_paths.append(os.path.join(root, file))
    return file_paths

# NOTE Change the project directory here
file_folder_path = os.path.join(os.getcwd(), "policy_vcm_docs")  # Adjust this to your actual path
print(f"file path is {file_folder_path}")

chroma_collection_policy = chroma_client.get_or_create_collection(name="vcm_policy")# construct vector store and customize storage context
storage_context_policy = StorageContext.from_defaults(
    vector_store=ChromaVectorStore(chroma_collection_policy)
)

# Function to check if file has already been processed
def file_already_processed(collection, file_name):
    """Check if a file has already been processed by searching collection metadata."""
    try:
        # Get all metadata from the collection
        all_metadata = collection.get(
            where={"file_name": file_name}
        )
        # If there are any results with this file_name, the file was already processed
        return len(all_metadata['metadatas']) > 0
    except Exception as e:
        print(f"Error checking if file was processed: {e}")
        return False

for file_path in get_file_paths(file_folder_path):
    file_name = os.path.basename(file_path)
    
    # Check if the file has already been processed
    if file_already_processed(chroma_collection_policy, file_name):
        print(f"File {file_name} already processed, skipping...")
        continue
    # Load documents
    documents = reader.load_data(Path(file_path))

    # Add file-based metadata
    for doc in documents:
        if doc.metadata is None:
            doc.metadata = {}
        doc.metadata.update({
            "file_name": file_name,
            "source": file_name.split("_")[0],
            "type": "policy_vcm"
        })
        
    index = VectorStoreIndex.from_documents(
        documents=documents,
        transformations=[node_parser],
        storage_context=storage_context_policy,
        cache=IngestionCache()
    )

    print(f"Successfully indexed with metadata")

INFO:docling.document_converter:Going to convert document batch...
INFO:docling.pipeline.base_pipeline:Processing document CAR_Forest_V5.0_Summary_for_Landowners.pdf


file path is /Users/beckyxu/Documents/GitHub/Carbon_Offset_Validation/carbon-offset-validator/server-python/policy_vcm_docs


INFO:docling.document_converter:Finished converting document CAR_Forest_V5.0_Summary_for_Landowners.pdf in 1.08 sec.
Batches: 100%|██████████| 1/1 [00:00<00:00,  4.07it/s]
INFO:docling.document_converter:Going to convert document batch...
INFO:docling.pipeline.base_pipeline:Processing document ICVCM_CCPs.pdf


Successfully indexed with metadata


INFO:docling.document_converter:Finished converting document ICVCM_CCPs.pdf in 132.08 sec.
Batches: 100%|██████████| 1/1 [00:00<00:00,  7.86it/s]
INFO:docling.document_converter:Going to convert document batch...
INFO:docling.pipeline.base_pipeline:Processing document VERRA_VT0009-Combined-Baseline-and-Additionality-Assessment-v1.0.pdf


...

Successfully indexed with metadata


INFO:docling.document_converter:Finished converting document CAR_Final_Forest_Protocol_V5.1_7.14.2023.pdf in 285.29 sec.
Batches: 100%|██████████| 1/1 [00:00<00:00,  1.38it/s]
INFO:docling.document_converter:Going to convert document batch...
INFO:docling.pipeline.base_pipeline:Processing document VERRA_Tool_demonstration_and_assessment_of_additionality .pdf


Successfully indexed with metadata


INFO:docling.document_converter:Finished converting document VERRA_Tool_demonstration_and_assessment_of_additionality .pdf in 10.06 sec.
Batches: 100%|██████████| 1/1 [00:00<00:00,  2.13it/s]

Successfully indexed with metadata


In [ ]:
# Retreive Index 
# chroma_client = chromadb.PersistentClient(path="./chroma_db_new")

# chroma_collection_policy = chroma_client.get_or_create_collection(name="vcm_policy")
# vector_store_policy = ChromaVectorStore(chroma_collection_policy)

# index = VectorStoreIndex.from_vector_store(vector_store=vector_store_policy)
# query_engine_policy = index.as_query_engine(similarity_top_k=4)

# # Change the code to compare PDD  index against the policy docs index and identify additionality risks
# # response = query_engine_policy.query("What is the carbon market?")
# # print(response)

In [228]:
# Check number of results
chroma_collection_pdd = chroma_client.get_or_create_collection(name="pdd")
result = chroma_collection_pdd.get()

nodes = []
if result and 'ids' in result and len(result['ids']) > 0:
    for i, node_id in enumerate(result['ids']):
        node_info = {
            'id': node_id,
            'text': result['documents'][i] if 'documents' in result else None,
            'metadata': result['metadatas'][i] if 'metadatas' in result else None
        }
        nodes.append(node_info)
        
print(f"Retrieved {len(nodes)} nodes from collection")


Retrieved 115 nodes from collection


#### Delete ChromaCollection

In [203]:
# collections = chroma_client.list_collections()
# # Delete each collection
# for collection in collections:
#     chroma_client.delete_collection(collection)
#     print(f"Deleted collection: {collection}")
# collections

Deleted collection: policy_vcm_docs
Deleted collection: vcm_policy


['policy_vcm_docs', 'vcm_policy']

# 2. LLM Service

#### Global Setting

In [215]:
# Test the request"
import requests

API_URL = "http://localhost:11434/api/generate"  # Default to Ollama API endpoint

payload = {
    "model": "deepseek-r1:7b",
    "prompt": "Hello, how are you?",
    "temperature": 0.2,
    "max_tokens": 50,
    "stream": False
}

headers = {
    "Content-Type": "application/json"
}

response = requests.post(API_URL, headers=headers, json=payload)

print("Status Code:", response.status_code)
print("Response:", response.text)


Status Code: 200
Response: {"model":"deepseek-r1:7b","created_at":"2025-03-30T20:20:56.796043Z","response":"\u003cthink\u003e\nAlright, someone just said \"Hello\" to me. That's a nice start! I should respond warmly to make them feel welcome.\n\nI want to let them know I'm here to help with whatever they need.\n\nMaybe say something like, \"I'm doing well, thank you! How can I assist you today?\"\n\nThat sounds friendly and opens the conversation for them to share what's on their mind.\n\u003c/think\u003e\n\nHello! I'm doing well. How can I assist you today?","done":true,"done_reason":"stop","context":[151644,9707,11,1246,525,498,30,151645,151648,198,71486,11,4325,1101,1053,330,9707,1,311,752,13,2938,594,264,6419,1191,0,358,1265,5889,96370,311,1281,1105,2666,10565,382,40,1366,311,1077,1105,1414,358,2776,1588,311,1492,448,8820,807,1184,382,21390,1977,2494,1075,11,330,40,2776,3730,1632,11,9702,498,0,2585,646,358,7789,498,3351,11974,4792,10362,11657,323,15885,279,10435,369,1105,311,4332,1

In [7]:
# helper function:
import requests
import xml.etree.ElementTree as ET
from typing import Dict, Any
import google.generativeai as genai

API_URL = os.getenv("LLM_API_URL", "http://localhost:11434/api/generate") 
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)

def call_llm_api(prompt: str) -> str:
    """
    Call Gemini LLM API with the provided prompt
    Args:
        prompt (str): The prompt to send to the LLM
    Returns:
        str: The LLM's response text
    """
    try:
        # Initialize the Gemini model (adjust model name as needed)
        model = genai.GenerativeModel(
            model_name="gemini-2.0-flash",  # Use "gemini-1.5-flash" or another available model
            generation_config={
                "temperature": 0.1,        # Controls randomness (0.0 to 1.0)
                "max_output_tokens": 50000, # Max tokens in response
            }
        )

        # Generate content with the prompt
        response = model.generate_content(prompt)

        # Check if response was blocked or empty
        if not response.text:
            raise Exception("Gemini API returned no valid response")

        return response.text

    except Exception as e:
        raise Exception(f"Gemini API error: {str(e)}")
    # API_URL = "http://localhost:11434/api/generate"
    # def call_llm_api(prompt: str) -> str:
    #     """
    #     Call LLM API with the provided prompt
    #     """
    #     headers = {
    #         "Content-Type": "application/json"
    #     }
        
    #     payload = {
    #         "model": "deepseek-r1:7b",
    #         "prompt": prompt,
    #         "temperature": 0.2,
    #         "max_tokens": 50000,
    #         "stream": False
    #     }
        
    #     response = requests.post(API_URL, headers=headers, json=payload)
        
    #     if response.status_code != 200:
    #         raise Exception(f"LLM API error: {response.status_code} {response.text}")
        
    #     # Extract the content from Ollama response
    #     return response.json()["response"]

def parse_xml_response(response: str, root_tag: str) -> Dict[str, Any]:
    """
    Parse XML response from LLM into a dictionary, handling potential extra text.
    
    Args:
        response (str): Raw LLM response containing XML
        root_tag (str): Expected root tag (e.g., "project_info")
    
    Returns:
        Dict[str, Any]: Parsed XML as a dictionary
    """
    # Try to find any XML-like structure if the exact root_tag isn't found
    xml_start = response.find(f"<{root_tag}>")
    xml_end = response.rfind(f"</{root_tag}>")
    
    if xml_start == -1 or xml_end == -1:
        # Fallback: Look for any XML root tag (e.g., <information>)
        possible_start = response.find("<")
        possible_end = response.rfind(">")
        if possible_start != -1 and possible_end != -1 and possible_end > possible_start:
            xml_content = response[possible_start:possible_end + 1]
        else:
            raise Exception("Could not find XML in LLM response")
    else:
        xml_content = response[xml_start:xml_end + len(f"</{root_tag}>")]

    # Parse XML
    try:
        root = ET.fromstring(xml_content)
    except ET.ParseError as e:
        raise Exception(f"Invalid XML in LLM response: {e}")

    # Convert to dictionary recursively
    def xml_to_dict(element):
        result = {}
        # Handle attributes
        if element.attrib:
            result["@attributes"] = element.attrib
        # Handle children
        for child in element:
            child_data = xml_to_dict(child)
            if child.tag in result:
                if not isinstance(result[child.tag], list):
                    result[child.tag] = [result[child.tag]]
                result[child.tag].append(child_data)
            else:
                result[child.tag] = child_data
        # Handle text content
        text = element.text.strip() if element.text else ""
        if text and not result:
            return text
        elif text:
            result["#text"] = text
        return result

    parsed_dict = xml_to_dict(root)
    
    # If root tag doesn't match expected, warn but proceed
    if root.tag != root_tag:
        print(f"Warning: Expected root tag '{root_tag}', found '{root.tag}'. Proceeding with parsed data.")
    
    return parsed_dict


## Function A: PDD Project Basic Info Extraction

In [44]:
def extract_doc_basicInfo(project_code: str) -> Dict[str, Any]:
    """
    Extract basic project information from document using LLM with XML-formatted output
    Args:
        index_pdd (VectorStoreIndex): Index of the document to extract information from
        project_code (str): project code 
    Returns:
        Dictionary containing extracted project information
    """
    # context = f"\n\nAdditional context:\n{additional_context}" if additional_context else ""
    # Create a retriever
    # chroma_client = chromadb.PersistentClient(path="./chroma_db_new")
    # pdd_collection = chroma_client.get_or_create_collection("chroma_collection_pdd")
    # pdd_vector_store = ChromaVectorStore(chroma_collection=pdd_collection)
    # index_pdd = VectorStoreIndex.from_vector_store(vector_store=pdd_vector_store)
    
    chroma_client = chromadb.PersistentClient(path="./chroma_db_new")
    pdd_collection = chroma_client.get_or_create_collection("pdd")
    pdd_vector_store = ChromaVectorStore(chroma_collection=pdd_collection)
    index_pdd = VectorStoreIndex.from_vector_store(vector_store=pdd_vector_store)

    retriever = index_pdd.as_retriever(similarity_top_k=20, metadata_filters={"project_code": project_code})  # Retrieve top 10 most relevant chunks
    
    query = "Please extract the following details from the provided voluntary carbon market project design document: Project's name; Brief description of the project;Location of the project (e.g., country, region);Current project status (e.g., under development, operational, completed);Project start date;Project end date;Project methodology (e.g., specific carbon offset standard or protocol used);Project size (e.g., area in hectares or total carbon credits generated)"
    retrieved_nodes = retriever.retrieve(query)
    
    # Format retrieved documents into context
    retrieved_texts = "\n\n".join([node.text for node in retrieved_nodes])
    # print(retrieved_texts)
    prompt = f"""
    Please extract the following details from the provided voluntary carbon market project design document: Project's name; Brief description of the project; Location of the project (e.g., country, region);Current project status (e.g., under development, operational, completed);Project start date;Project end date;Project methodology (e.g., specific carbon offset standard or protocol used);Project size (e.g., area in hectares or total carbon credits generated)

    ### Instructions ###
    - Extract these exact fields from the document: project name, description, location, coordinates, status, start date, end date, methodology, size.
    - Use the *exact* XML tag names as shown in the Output Format: <name>, <description>, <location>, <status>, <start_date>, <end_date>, <methodology>, <size>.
    - Output *ONLY* the XML structure—do not include any additional text, comments, `<think>` tags, markdown (```xml```), or explanations before or after the XML.
    - Wrap the output in the root tag `<project_info>`.
    - Keep <project_code> as it is 
    - If a field is missing or not found, use "Not specified" as the value. Infer project name if project name is not found.
    - Ensure the XML is well-formed and matches the Output Format exactly in structure and tag names.

    ### Output Format ###
    <project_info>
      <project_code>{project_code}<project_code>
      <name>PROJECT TITLE</name>
      <description>BRIEF DESCRIPTION</description>
      <location>LOCATION</location>
      <status>STATUS</status>
      <start_date>START DATE</start_date>
      <end_date>END DATE</end_date>
      <methodology>METHODOLOGY</methodology>
      <size>SIZE</size>
    </project_info>

    ### Document to Analyze ###
    {retrieved_texts}

    ### Final Directive ###
    Return ONLY the XML below, using the exact tag names from the Output Format, with no deviations or additional content.
    """
    
    # Call your preferred LLM API
    response = call_llm_api(prompt)
    # print(response)
    # Parse XML response
    try:
        parsed_data = parse_xml_response(response, "project_info")
        return parsed_data
    except Exception as e:
        print(f"Error parsing LLM response: {e}")
        raise Exception(f"Failed to parse LLM output: {e}")


#### Need to get vectorstoreindex to use the function

In [229]:

# chroma_client = chromadb.PersistentClient(path="./chroma_db_new")
# pdd_collection = chroma_client.get_or_create_collection("pdd")
# pdd_vector_store = ChromaVectorStore(chroma_collection=pdd_collection)
# index_pdd = VectorStoreIndex.from_vector_store(vector_store=pdd_vector_store)
# # manual function running 
# retriever = index_pdd.as_retriever(similarity_top_k=20,metadata_filters={"project_code": '3226'})  # Retrieve top 10 most relevant chunks
# # 
# query = "Find project name, description, location, coordinates, status, start date, end date, methodology, and size from the project design document."
# retrieved_nodes = retriever.retrieve(query)

# # Format retrieved documents into context
# retrieved_texts = "\n\n".join([node.text for node in retrieved_nodes])

# print(retrieved_texts)

#### Result

In [45]:
# Result
project_data = extract_doc_basicInfo(project_code='3226')
project_data

```xml
<project_info>
  <project_code>3226</project_code>
  <name>Padang Tikar Landscape REDD+ Project</name>
  <description>REDD+ project focused on reducing GHG emissions from unplanned deforestation and wetland degradation through conservation and sustainable management activities.</description>
  <location>Padang Tikar Landscape, Kubu Raya, West Kalimantan, Indonesia</location>
  <status>Under development</status>
  <start_date>August 30, 2017</start_date>
  <end_date>August 29, 2047</end_date>
  <methodology>VM0007 REDD+ Methodology Framework (REDD+MF), Version 1.6</methodology>
  <size>58,672.7 ha</size>
</project_info>
```


{'project_code': '3226',
 'name': 'Padang Tikar Landscape REDD+ Project',
 'description': 'REDD+ project focused on reducing GHG emissions from unplanned deforestation and wetland degradation through conservation and sustainable management activities.',
 'location': 'Padang Tikar Landscape, Kubu Raya, West Kalimantan, Indonesia',
 'status': 'Under development',
 'start_date': 'August 30, 2017',
 'end_date': 'August 29, 2047',
 'methodology': 'VM0007 REDD+ Methodology Framework (REDD+MF), Version 1.6',
 'size': '58,672.7 ha'}

## Function B: PDD vs VCM_Policy Risk Analysis

In [17]:
# Change model to gemini
llm_g, _ = llm_setting("Gemini")
# Set default LLM and embedding model 
# Settings.llm = llm

/var/folders/j2/yjk_0cz112g3l8vv2_013tmr0000gn/T/ipykernel_1286/3485569342.py:16: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/)
  llm = Gemini(model_name="models/gemini-1.5-flash", temperature=0.1, max_tokens=50000, api_key=api_key)


In [63]:
import chromadb
from llama_index.core import VectorStoreIndex, Settings, PromptTemplate
from llama_index.core.retrievers import BaseRetriever
from typing import List, Dict, Any
import xml.etree.ElementTree as ET


def analyze_project_risks(project_code: str, top_k: int = 5) -> List[Dict[str, Any]]:
    """
    Analyze project risks by comparing PDD against policy documents, returning XML-structured results.

    Args:
        project_code: project code 
        top_k: Number of top documents to retrieve.

    Returns:
        List of dictionaries containing risk metrics per query.
    """
    # Setup Chroma connections
    chroma_client = chromadb.PersistentClient(path="./chroma_db_new")
    pdd_collection = chroma_client.get_or_create_collection("pdd")
    policy_collection = chroma_client.get_or_create_collection("vcm_policy")

    # Create vector stores and indexes
    from llama_index.vector_stores.chroma import ChromaVectorStore
    pdd_vector_store = ChromaVectorStore(chroma_collection=pdd_collection)
    policy_vector_store = ChromaVectorStore(chroma_collection=policy_collection)
    pdd_index = VectorStoreIndex.from_vector_store(vector_store=pdd_vector_store)
    policy_index = VectorStoreIndex.from_vector_store(vector_store=policy_vector_store)

    # Custom dual retriever
    class DualRetriever(BaseRetriever):
        def __init__(self, pdd_retriever, policy_retriever):
            self.pdd_retriever = pdd_retriever
            self.policy_retriever = policy_retriever
            super().__init__()

        def _retrieve(self, query, **kwargs):
            pdd_nodes = self.pdd_retriever.retrieve(query)
            policy_nodes = self.policy_retriever.retrieve(query)
            # Add metadata to nodes
            for node in pdd_nodes:
                node.node.metadata["source"] = "project_design_document"
            for node in policy_nodes:
                node.node.metadata["source"] = "industry_standard"
            # Return a single list combining both sets of nodes
            return pdd_nodes + policy_nodes

    pdd_retriever = pdd_index.as_retriever(similarity_top_k=top_k, metadata_filters={"project_code": project_code})
    policy_retriever = policy_index.as_retriever(similarity_top_k=top_k)
    dual_retriever = DualRetriever(pdd_retriever, policy_retriever)

    # Custom prompt for XML output
    risk_template_str = (
        "Analyze the risk profile of a carbon offset project by comparing its project design document (PDD) "
        "with established carbon offset policy documents for the aspect: {query}.\n\n"
        "<instructions>\n"
        "- Review the PDD content: {pdd_texts}\n"
        "- Compare it against policy standards: {policy_texts}\n"
        "- Identify potential risks in the category listed in < > in query.\n"
        "- Assign an overall risk score (0-100), impact level (Low, Medium, High), and likelihood (Unlikely, Possible, Likely).\n"
        "- Provide a brief description for the risks.\n"
        "- Provide a list of keywords, separated by comma, that describe the risks\n"
        "- Use the *exact* XML tag names as listed in after query's analyze the risk category:.\n"
        "- Output *ONLY* one <risk_category>\. If there are multiple risks, then explain in the description.\n"
        "- Output *ONLY* the XML structure—do not include additional text, comments, `<think>` tags, markdown, or explanations.\n"
        "</instructions>\n\n"
        "<output_format>\n"
        "<risk_metrics>\n"
        "  <risk_category name=\"CATEGORY\">\n"
        "    <score>SCORE_VALUE</score>\n"
        "    <impact>IMPACT_LEVEL</impact>\n"
        "    <likelihood>LIKELIHOOD</likelihood>\n"
        "    <description>RISK_DESCRIPTION</description>\n"
        "    <keywords>RISK_KEYWORDS</keywords>\n"
        "  </risk_category>\n"
        "</risk_metrics>\n"
        "</output_format>"
    )
    risk_template = PromptTemplate(risk_template_str)

    all_risk_metrics = []
    
    project_query_list = ["analyze the risk category: <Additionality> - How does the project demonstrate that it is additional, and what evidence supports this claim? Additionality ensures that the emissions reductions or removals would not have occurred without carbon offset funding. Look for evidence such as financial barriers, technological challenges, or policy gaps that the project overcomes.",
                      "analyze the risk category: <Baseline Scenario> - What is the baseline scenario for the project, and how was it established? The baseline scenario represents the emissions that would have occurred without the project. Check if it’s based on credible data, conservative assumptions, and an appropriate methodology for the project type.",
                      "analyze the risk category: <Permanence> - For projects involving carbon sequestration, what measures are in place to ensure the permanence of the sequestered carbon? For projects like reforestation or soil carbon storage, permanence is critical. Ask about safeguards like buffer pools, long-term management plans, or insurance against reversals (e.g., due to fires or deforestation).",
                      "analyze the risk category: <Leakage> - Has the project assessed potential leakage, and how is it accounted for in the emissions reductions calculations? Leakage occurs when emissions are displaced elsewhere (e.g., deforestation shifting to another area). Verify if a leakage assessment was conducted and if mitigation measures are included.",
                      "analyze the risk category: <Monitoring and Verification> - What is the monitoring plan, and how will the project's emissions reductions be verified by a third party? A robust monitoring plan should detail what data will be collected, how, and how often. Confirmation of third-party verification ensures accuracy and independence."
                     ]
    
    # Process each query using risk_query_engine
    for query in project_query_list:
        print(f"Analyzing: {query}")
        # Retrieve nodes using the dual retriever
        nodes = dual_retriever.retrieve(query)
        # Separate nodes by source for context
        pdd_nodes = [node for node in nodes if node.node.metadata.get("source") == "project_design_document"]
        policy_nodes = [node for node in nodes if node.node.metadata.get("source") == "industry_standard"]
        # Combine retrieved content into context
        pdd_context_str = "\n\n".join([node.node.get_content() for node in pdd_nodes])
        policy_context_str = "\n\n".join([node.node.get_content() for node in policy_nodes])
        # Format the full prompt with context
        formatted_prompt = risk_template.format(
            query=query,
            project_code=project_code,
            pdd_texts=pdd_context_str,
            policy_texts=policy_context_str
        )
        # Call the LLM directly
        response_text = call_llm_api(formatted_prompt)  # Assuming this function is defined elsewhere
        try:
            # Parse XML using the new function
            parsed_response = parse_xml_response(response_text, "risk_metrics")
            risk_metrics = []
            
            # Handle case where risk_category is a single dict or a list
            risk_categories = parsed_response.get("risk_category")
            if not risk_categories:
                print(f"No risk categories found for '{query}'")
                continue
            if isinstance(risk_categories, dict):
                risk_categories = [risk_categories]
                
            for category in risk_categories:
                risk_metrics.append({
                    "category": category["@attributes"]["name"],
                    "score": int(category["score"]),
                    "impact": category["impact"],
                    "likelihood": category["likelihood"],
                    "description": category["description"],
                    # "query": query
                })
            all_risk_metrics.extend(risk_metrics)
            # print(f"Risk metrics for '{query}': {response_text}")
        except Exception as e:
            print(f"Error parsing risk metrics for '{query}': {e}")
            raise Exception(f"Failed to parse risk metrics: {e}")

    return all_risk_metrics

<>:65: SyntaxWarning: invalid escape sequence '\.'
<>:65: SyntaxWarning: invalid escape sequence '\.'
/var/folders/j2/yjk_0cz112g3l8vv2_013tmr0000gn/T/ipykernel_1286/370823593.py:65: SyntaxWarning: invalid escape sequence '\.'
  "- Output *ONLY* one <risk_category>\. If there are multiple risks, then explain in the description.\n"


#### Result

In [65]:
project_code = '3226'
all_risk_metrics = analyze_project_risks(project_code)
all_risk_metrics

Analyzing: analyze the risk category: <Additionality> - How does the project demonstrate that it is additional, and what evidence supports this claim? Additionality ensures that the emissions reductions or removals would not have occurred without carbon offset funding. Look for evidence such as financial barriers, technological challenges, or policy gaps that the project overcomes.
Analyzing: analyze the risk category: <Baseline Scenario> - What is the baseline scenario for the project, and how was it established? The baseline scenario represents the emissions that would have occurred without the project. Check if it’s based on credible data, conservative assumptions, and an appropriate methodology for the project type.
Analyzing: analyze the risk category: <Permanence> - For projects involving carbon sequestration, what measures are in place to ensure the permanence of the sequestered carbon? For projects like reforestation or soil carbon storage, permanence is critical. Ask about saf

[{'category': 'Additionality',
  'score': 65,
  'impact': 'High',
  'likelihood': 'Possible',
  'description': "The project's additionality relies on demonstrating that conservation would not occur without carbon credit financing due to historical non-compliance with conservation measures, insufficient economic incentives for local communities, and poverty levels. The risk lies in the potential overestimation of the baseline deforestation rate or underestimation of alternative conservation efforts that could occur independently. While the PDD mentions the use of tools for demonstrating additionality, the strength of the evidence and the rigor of applying these tools are critical. Policy documents emphasize the need to demonstrate that reductions are above and beyond legal requirements and business-as-usual activities. A key risk is that the project area might have been subject to conservation efforts regardless, or that the economic barriers are not as significant as claimed. The relia

## Function C: PDD vs Regional Policy Risk Analysis

#### Result - simulated 

In [66]:
# Result - Simulated:
risk_policy = {
    "summary": {
        "overall_summary": "This is a summary of the project risks and compliance with policies.",
        "recommendations": [
            {"action": "Strengthen additionality evidence"},
            {"action": "Improve leakage monitoring"},
            {"action": "Enhance permanence safeguards"}
        ],
        "additional_insights": "Additional insights about the project..."
    }
}

In [67]:
from database import store_analysis_results
async def upload_to_supabase():
    project_id = await store_analysis_results(project_data, all_risk_metrics, risk_policy)
    print(f"Successfully uploaded project data with ID: {project_id}")
    # Run the async function
asyncio.run(upload_to_supabase())

Successfully uploaded project data with ID: 084650fe-b1f5-44d4-9a40-a7d4e7f42f30
